# Calibration Notebook

Orchestrates PM (Model C — XGBoost shape classifier) and TM engines across
the test set, buckets predicted probabilities, computes actual occurrence rates,
and derives USE/SOFT/IGNORE thresholds.

**PM = Model C** (predicts current shape from fundamentals, ~90% OOS)
NOT the persistence engine (which predicts binary 4w/12w persistence).

**TM 1w = XGBoost regularised Config B** (swapped from RF July 2026)
**TM 2w = XGBoost regularised Config B** (unchanged)

In [1]:
import sys
sys.path.insert(0, r'C:/ClaudeCode')

import pandas as pd
import numpy as np
from datetime import datetime
from models.pm_engine import predict as pm_predict
from models.tm_engine import predict as tm_predict

LOG_FILE = r'C:/ClaudeCode/research/04. backtest_analysis/backtest_analysis.txt'
TRAIN_END = '2024-12-31'
TEST_START = '2025-01-01'
CI_THRESHOLD = 30  # buckets with fewer than this many observations get 'wide CI' flag

print('Imports OK — PM = Model C (XGBoost shape classifier)')

Imports OK — PM = Model C (XGBoost shape classifier)


In [2]:
# ── Run PM (Model C) predictions across test set ──────────────
from models.feature_prep import build_tm_daily_panel, TM_FEATURES

daily = build_tm_daily_panel()
test_daily = daily[daily['date'] > TRAIN_END].dropna(
    subset=TM_FEATURES + ['shape']).copy()

pm_results = []
for _, row in test_daily.iterrows():
    dt = row['date']
    try:
        pred = pm_predict(dt)
        pm_results.append(pred)
    except Exception as e:
        print(f'PM error at {dt.date()}: {e}')

pm_df = pd.DataFrame(pm_results)
pm_date_min = test_daily['date'].min().date()
pm_date_max = test_daily['date'].max().date()
print(f'PM predictions: {len(pm_df)} rows')
print(f'Date range: {pm_date_min} to {pm_date_max}')
print(f'Overall accuracy: {(pm_df["predicted_shape"] == pm_df["observed_shape"]).mean()*100:.1f}%')
pm_df.head()

PM predictions: 360 rows
Date range: 2025-01-02 to 2026-06-26
Overall accuracy: 89.4%


,predicted_shape,confidence,shape_probs,observed_shape,date
0,0.0,0.997837,"{'0.0': 0.997837245464325, '0.1': 0.0012197764...",0.0,2025-01-02
1,0.0,0.997594,"{'0.0': 0.9975935816764832, '0.1': 0.001090877...",0.0,2025-01-03
2,0.0,0.997745,"{'0.0': 0.9977445602416992, '0.1': 0.000923427...",0.0,2025-01-06
3,0.0,0.997284,"{'0.0': 0.9972844123840332, '0.1': 0.001137698...",0.0,2025-01-07
4,0.0,0.997592,"{'0.0': 0.9975923895835876, '0.1': 0.001045367...",0.0,2025-01-08


In [3]:
# ── Run TM predictions across test set dates ──────────────────
test_daily_tm = daily[daily['date'] > TRAIN_END].dropna(
    subset=TM_FEATURES + ['target_1w', 'target_2w']).copy()

tm_results = []
for _, row in test_daily_tm.iterrows():
    dt = row['date']
    for hz in ['1w', '2w']:
        try:
            pred = tm_predict(dt, horizon=hz)
            target_col = 'target_1w' if hz == '1w' else 'target_2w'
            pred['actual_next_shape'] = row[target_col]
            pred['correct'] = pred.get('top1_shape') == row[target_col]
            tm_results.append(pred)
        except Exception as e:
            print(f'TM error at {dt.date()} {hz}: {e}')

tm_df = pd.DataFrame(tm_results)
tm_date_min = test_daily_tm['date'].min().date()
tm_date_max = test_daily_tm['date'].max().date()
print(f'TM predictions: {len(tm_df)} rows ({len(tm_df)//2} dates x 2 horizons)')
print(f'Date range: {tm_date_min} to {tm_date_max}')
tm_df.head()

TM predictions: 700 rows (350 dates x 2 horizons)
Date range: 2025-01-02 to 2026-06-11


,current_shape,date,horizon,top1_shape,top1_prob,all_probs,ruled_out,top2_shape,top2_prob,actual_next_shape,correct
0,0.0,2025-01-02,1w,0.0,0.8298,"{'0.0': 0.8298, '0.1': 0.1455, '0.2': 0.0186, ...","[0.2, 1, 2]",0.1,0.1455,0.0,True
1,0.0,2025-01-02,2w,0.0,0.8720,"{'0.0': 0.872, '0.1': 0.1019, '0.2': 0.0145, '...","[0.2, 1, 2]",0.1,0.1019,0.0,True
2,0.0,2025-01-03,1w,0.0,0.8352,"{'0.0': 0.8352, '0.1': 0.1405, '0.2': 0.0183, ...","[0.2, 1, 2]",0.1,0.1405,0.0,True
3,0.0,2025-01-03,2w,0.0,0.8817,"{'0.0': 0.8817, '0.1': 0.0921, '0.2': 0.0148, ...","[0.2, 1, 2]",0.1,0.0921,0.0,True
4,0.0,2025-01-06,1w,0.0,0.8358,"{'0.0': 0.8358, '0.1': 0.14, '0.2': 0.0182, '1...","[0.2, 1, 2]",0.1,0.1400,0.0,True


In [4]:
# ── Calibration analysis + threshold derivation ───────────────

changes_vs_prior = [
    '1w model swapped from RF (46.6%) to XGBoost Config B (65.7%). All other models unchanged.',
    'Both TM horizons now use XGBoost regularised (Config B) with same hyperparameter philosophy.',
    'See models/MODEL_REFERENCE.txt for full model documentation.',
]

output_lines = [
    f'CALIBRATION ANALYSIS (post 1w model swap to XGB) — {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    '=' * 70,
    '',
    'Changes vs prior run (2026-07-10 09:16):',
]
for c in changes_vs_prior:
    output_lines.append(f'  - {c}')

# ── TM calibration ────────────────────────────────────────────
for hz in ['1w', '2w']:
    sub = tm_df[tm_df['horizon'] == hz].copy()
    if len(sub) == 0:
        continue

    output_lines.append(f'\n--- TM {hz} CALIBRATION ---')
    output_lines.append(f'Date range: {tm_date_min} to {tm_date_max}')
    output_lines.append(f'Total predictions: {len(sub)} (daily, one per trading day)')
    output_lines.append(f'Filter: test set dates with both target_1w and target_2w non-null')
    output_lines.append(f'Overall top-1 accuracy: {sub["correct"].mean()*100:.1f}%')

    # Bucket by top1_prob
    bins = [0, 0.3, 0.5, 0.7, 1.01]
    labels = ['<30%', '30-50%', '50-70%', '70%+']
    sub['prob_bucket'] = pd.cut(sub['top1_prob'], bins=bins, labels=labels)

    output_lines.append(f'\n  {"Bucket":<10} {"N":>5} {"Accuracy":>10} {"CI Note":>10}')
    output_lines.append(f'  {"-"*40}')
    for bucket in labels:
        bucket_data = sub[sub['prob_bucket'] == bucket]
        n = len(bucket_data)
        if n == 0:
            output_lines.append(f'  {bucket:<10} {0:>5} {"—":>10} {"no data":>10}')
            continue
        acc = bucket_data['correct'].mean() * 100
        ci_note = 'wide CI' if n < CI_THRESHOLD else ''
        output_lines.append(f'  {bucket:<10} {n:>5} {acc:>9.1f}% {ci_note:>10}')

    # Per current_shape breakdown
    output_lines.append(f'\n  Per-shape accuracy:')
    for shape in sorted(sub['current_shape'].unique()):
        shape_sub = sub[sub['current_shape'] == shape]
        acc = shape_sub['correct'].mean() * 100
        output_lines.append(
            f'    Shape {shape}: {acc:.1f}% (n={len(shape_sub)})')

# ── PM (Model C) calibration ──────────────────────────────────
output_lines.append(f'\n--- PM CALIBRATION (Model C — XGBoost shape classifier) ---')
output_lines.append(f'Date range: {pm_date_min} to {pm_date_max}')
output_lines.append(f'Total predictions: {len(pm_df)} (daily, one per trading day)')
output_lines.append(f'Filter: test set dates with all 11 TM features non-null')

pm_df['correct'] = pm_df['predicted_shape'] == pm_df['observed_shape']
pm_acc = pm_df['correct'].mean() * 100
output_lines.append(f'Overall accuracy: {pm_acc:.1f}%')

# Bucket by confidence
bins_pm = [0, 0.5, 0.7, 0.9, 1.01]
labels_pm = ['<50%', '50-70%', '70-90%', '90%+']
pm_df['conf_bucket'] = pd.cut(pm_df['confidence'], bins=bins_pm, labels=labels_pm)

output_lines.append(f'\n  {"Bucket":<10} {"N":>5} {"Accuracy":>10} {"CI Note":>10}')
output_lines.append(f'  {"-"*40}')
for bucket in labels_pm:
    bucket_data = pm_df[pm_df['conf_bucket'] == bucket]
    n = len(bucket_data)
    if n == 0:
        output_lines.append(f'  {bucket:<10} {0:>5} {"—":>10} {"no data":>10}')
        continue
    acc = bucket_data['correct'].mean() * 100
    ci_note = 'wide CI' if n < CI_THRESHOLD else ''
    output_lines.append(f'  {bucket:<10} {n:>5} {acc:>9.1f}% {ci_note:>10}')

output_lines.append(f'\n  Per-shape accuracy:')
for shape in sorted(pm_df['observed_shape'].unique()):
    shape_sub = pm_df[pm_df['observed_shape'] == shape]
    acc = shape_sub['correct'].mean() * 100
    output_lines.append(f'    Shape {shape}: {acc:.1f}% (n={len(shape_sub)})')

# ── Disagreement analysis (PM predicted vs observed) ──────────
output_lines.append(f'\n  PM disagreement (predicted != observed):')
disagree = pm_df[~pm_df['correct']]
output_lines.append(f'    Total disagreement days: {len(disagree)} of {len(pm_df)} ({len(disagree)/len(pm_df)*100:.1f}%)')
if len(disagree) > 0:
    output_lines.append(f'    Avg confidence on disagreement days: {disagree["confidence"].mean()*100:.1f}%')
    output_lines.append(f'    Avg confidence on agreement days: {pm_df[pm_df["correct"]]["confidence"].mean()*100:.1f}%')

# ── Derived thresholds ────────────────────────────────────────
output_lines.append(f'\n--- DERIVED THRESHOLDS ---')
output_lines.append('Gap = actual_occurrence_rate(high_prob_half) - actual_occurrence_rate(low_prob_half)')
output_lines.append('Positive gap = model discriminates correctly. USE: gap>5pp, SOFT: gap>2pp, IGNORE: gap<=2pp')

for hz in ['1w', '2w']:
    sub = tm_df[tm_df['horizon'] == hz].copy()
    if len(sub) == 0:
        continue

    output_lines.append(f'\n  {hz} horizon:')
    for shape in sorted(sub['current_shape'].unique()):
        shape_sub = sub[sub['current_shape'] == shape]

        all_probs_records = []
        for _, row in shape_sub.iterrows():
            ap = row.get('all_probs', {})
            if isinstance(ap, dict):
                for next_s, prob in ap.items():
                    all_probs_records.append({
                        'next_shape': next_s,
                        'prob': prob,
                        'actual': row['actual_next_shape'],
                        'occurred': int(row['actual_next_shape'] == next_s),
                    })

        if not all_probs_records:
            output_lines.append(f'    Shape {shape}: NO DATA (n={len(shape_sub)} test rows, 0 qualifying probability records)')
            continue

        prob_df = pd.DataFrame(all_probs_records)
        output_lines.append(f'    Shape {shape} (n={len(shape_sub)} test days):')

        any_output = False
        for next_s in sorted(prob_df['next_shape'].unique()):
            ns = prob_df[prob_df['next_shape'] == next_s]
            n_total = len(ns)

            if n_total < 5:
                output_lines.append(
                    f'      -> {next_s}: SKIPPED (n={n_total}, need >=5 qualifying predictions)')
                any_output = True
                continue

            median_prob = ns['prob'].median()
            high = ns[ns['prob'] > median_prob]
            low = ns[ns['prob'] <= median_prob]

            if len(high) < 3 or len(low) < 3:
                output_lines.append(
                    f'      -> {next_s}: SKIPPED (n={n_total}, but median split produced '
                    f'high={len(high)}/low={len(low)} — need >=3 each)')
                any_output = True
                continue

            high_rate = high['occurred'].mean()*100
            low_rate = low['occurred'].mean()*100
            gap = high_rate - low_rate
            rating = 'USE' if gap > 5 else ('SOFT' if gap > 2 else 'IGNORE')
            ci_note = '  [wide CI]' if n_total < CI_THRESHOLD else ''
            output_lines.append(
                f'      -> {next_s}: gap={gap:+.1f}pp  hi={high_rate:.1f}%({len(high)})  '
                f'lo={low_rate:.1f}%({len(low)})  rating={rating}{ci_note}')
            any_output = True

        if not any_output:
            output_lines.append(f'      (no qualifying transition cells)')

# Write to log
content = '\n'.join(output_lines)
print(content)

with open(LOG_FILE, 'a', encoding='utf-8') as f:
    f.write('\n' + content + '\n')
print(f'\n[Written to {LOG_FILE}]')

CALIBRATION ANALYSIS (post 1w model swap to XGB) — 2026-07-10 13:06

Changes vs prior run (2026-07-10 09:16):
  - 1w model swapped from RF (46.6%) to XGBoost Config B (65.7%). All other models unchanged.
  - Both TM horizons now use XGBoost regularised (Config B) with same hyperparameter philosophy.
  - See models/MODEL_REFERENCE.txt for full model documentation.

--- TM 1w CALIBRATION ---
Date range: 2025-01-02 to 2026-06-11
Total predictions: 350 (daily, one per trading day)
Filter: test set dates with both target_1w and target_2w non-null
Overall top-1 accuracy: 65.7%

  Bucket         N   Accuracy    CI Note
  ----------------------------------------
  <30%           0          —    no data
  30-50%        36      50.0%           
  50-70%       125      62.4%           
  70%+         189      70.9%           

  Per-shape accuracy:
    Shape 0.0: 82.5% (n=80)
    Shape 0.1: 48.0% (n=25)
    Shape 0.2: 14.3% (n=7)
    Shape 1: 62.2% (n=111)
    Shape 2: 64.6% (n=127)

--- TM 2w CA